# Introduction To Cohere


## Load Keys in the Env.


In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

## Initialize Cohere Client


In [2]:
import os
import cohere

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
co

## Text Generation


### Normal Chat


In [ ]:
response = co.chat(
    model="command-a-03-2025",
    messages=[
        {
            "role": "system",
            "content": "You are a concise teaching assistant. Answer in 2 sentences max.",
        },
        {
            "role": "user",
            "content": "In plain English, what is a vector embedding?",
        },
    ],
)

print(response.message.content[0].text)

A vector embedding is a numerical representation of data (like text, images, or audio) in a multi-dimensional space, capturing its meaning or features in a way that similar items are closer together. It allows machines to understand and compare complex data by converting it into a structured format.


In [4]:
response

V2ChatResponse(id='121cc761-0f5a-4d8d-a200-9d6108e463d8', finish_reason='COMPLETE', message=AssistantMessageResponse(role='assistant', tool_calls=None, tool_plan=None, content=[TextAssistantMessageResponseContentItem(type='text', text='A vector embedding is a numerical representation of data (like text, images, or audio) in a multi-dimensional space, capturing its meaning or features in a way that similar items are closer together. It allows machines to understand and compare complex data by converting it into a structured format.')], citations=None), usage=Usage(billed_units=UsageBilledUnits(input_tokens=24.0, output_tokens=57.0, search_units=None, classifications=None), tokens=UsageTokens(input_tokens=553.0, output_tokens=59.0), cached_tokens=480.0), logprobs=None)

In [5]:
print(f"Finish Reason: {response.finish_reason}")
print(f"Tokens Used: {response.usage.tokens if response.usage else 'n/a'}")
print(f"Role: {response.message.role}")

Finish Reason: COMPLETE
Tokens Used: input_tokens=553.0 output_tokens=59.0
Role: assistant


### Streaming Chat


In [ ]:
stream = co.chat_stream(
    model="command-a-03-2025",
    messages=[
        {
            "role": "user",
            "content": "Give me a 4-line haiku about debugging AI agents at 3 am.",
        },
    ],
)

for event in stream:
    if event.type == "content-delta":
        print(event.delta.message.content.text, end="", flush=True)

Midnight code whispers,  
Tracing ghosts in neural streams—  
A bug blinks awake.  

Caffeine fuels the hunt,  
Logic bends where silence breaks—  
Dawn meets fixed machine.

## Text Embeddings


### Cosine Similariy


In [8]:
docs = [
    "The cat sat on the mat.",
    "A feline rested on the rug.",
    "Quantum Entanglement links two particles across distance.",
]

doc_embeds = co.embed(
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
    texts=docs,
).embeddings.float

print(f"Got {len(doc_embeds)} embeddings, each of dimension {len(doc_embeds[0])}")

Got 3 embeddings, each of dimension 1536


In [ ]:
import math


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))

    return dot / (na * nb)


print(f"cat <-> feline : {cosine(doc_embeds[0], doc_embeds[1])}")
print(f"cat <-> quantum : {cosine(doc_embeds[0], doc_embeds[2])}")

cat <-> feline : 0.6060927509977355
cat <-> quantum : 0.318964023730803


### Mini Semantic Search


In [11]:
query = "animal is on a carpet"

query_embed = co.embed(
    model="embed-v4.0",
    input_type="search_query",
    embedding_types=["float"],
    texts=[query],
).embeddings.float[0]

scored = sorted(
    [
        (
            cosine(query_embed, d),
            text,
        )
        for d, text in zip(
            doc_embeds,
            docs,
        )
    ],
)

print(f"Query: {query!r}\n")
for score, text in scored:
    print(f"    {score} -- {text}")

Query: 'animal is on a carpet'

    0.11835334461867825 -- Quantum Entanglement links two particles across distance.
    0.3234774214571595 -- The cat sat on the mat.
    0.48033509139249747 -- A feline rested on the rug.


### Multi-modal embedding

In [13]:
import base64
import requests
from io import BytesIO
from PIL import Image

image_urls = {
    "cat": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400",
    "dog": "https://images.unsplash.com/photo-1558788353-f76d92427f16?w=400",
    "city": "https://images.unsplash.com/photo-1477959858617-67f85cf4f1df?w=400",
}


def url_to_data_uri(url: str) -> str:
    """Download an image and turn it into a base64 data URI for the API."""
    raw = requests.get(url, timeout=30).content
    img = Image.open(BytesIO(raw)).convert("RGB")
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


image_inputs = [
    {"content": [{"type": "image_url", "image_url": {"url": url_to_data_uri(u)}}]}
    for u in image_urls.values()
]

image_embeds = co.embed(
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
    inputs=image_inputs,
).embeddings.float

print(f"Embedded {len(image_embeds)} images, dim={len(image_embeds[0])}")

Embedded 3 images, dim=1536


In [14]:
text_query = "a fluffy animal"

text_query_embed = co.embed(
    model="embed-v4.0",
    input_type="search_query",
    embedding_types=["float"],
    texts=[text_query],
).embeddings.float[0]

print(f"Query: {text_query!r}\n")
for label, emb in zip(image_urls.keys(), image_embeds):
    print(f"  {label:5s} similarity = {cosine(text_query_embed, emb):.3f}")

Query: 'a fluffy animal'

  cat   similarity = 0.201
  dog   similarity = 0.213
  city  similarity = 0.038


In [17]:
text_query = "a busy urban skyline"

text_query_embed = co.embed(
    model="embed-v4.0",
    input_type="search_query",
    embedding_types=["float"],
    texts=[text_query],
).embeddings.float[0]

print(f"Query: {text_query!r}\n")
for label, emb in zip(image_urls.keys(), image_embeds):
    print(f"  {label:5s} similarity = {cosine(text_query_embed, emb):.3f}")

Query: 'a busy urban skyline'

  cat   similarity = 0.032
  dog   similarity = 0.043
  city  similarity = 0.361


## Re-Ranking

In [18]:
query = "How do I reset my password?"

documents = [
    "To change your account password, go to Settings → Security and click 'Reset password'.",
    "Our refund policy allows returns within 30 days of purchase.",
    "If you've forgotten your password, click 'Forgot password' on the login screen and follow the email link.",
    "We offer 24/7 customer support via chat and email.",
    "Two-factor authentication can be enabled from the Security settings page.",
]

reranked = co.rerank(
    model="rerank-v3.5",
    query=query,
    documents=documents,
    top_n=3,
)

print(f"Query: {query!r}\n")
print("Top 3 reranked results:")
for hit in reranked.results:
    print(f"  [{hit.relevance_score:.3f}] {documents[hit.index]}")

Query: 'How do I reset my password?'

Top 3 reranked results:
  [0.740] To change your account password, go to Settings → Security and click 'Reset password'.
  [0.548] If you've forgotten your password, click 'Forgot password' on the login screen and follow the email link.
  [0.173] Two-factor authentication can be enabled from the Security settings page.
